# 02 Feature Engineering

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
RESULTS = Path("../results")
PROCESSED.mkdir(parents=True, exist_ok=True)
CUTOFF = pd.Timestamp("2017-03-31")

train = pd.read_csv(RAW / "train_data.csv").drop_duplicates("msno", keep="last").reset_index(drop=True)
test = pd.read_csv(RAW / "kaggle_test_data.csv").drop_duplicates("msno").reset_index(drop=True)
users = pd.concat([train.msno, test.msno]).unique()
uid = pd.Series(np.arange(len(users), dtype=np.int32), index=users)  # integer id per msno, saves memory

## Member Features

In [2]:
members = pd.read_csv(RAW / "members.csv", dtype={"gender": "string"})
members = members[members.msno.isin(uid.index)].drop_duplicates("msno", keep="last")
reg_date = pd.to_datetime(members.registration_init_time.astype(str), format="%Y%m%d", errors="coerce")
member_feats = pd.DataFrame({
    "msno": members.msno.values,
    "city": members.city.values,
    "registered_via": members.registered_via.values,
    "gender": members.gender.map({"female": 1, "male": 2}).fillna(0).astype(int).to_numpy(),  # 0 = not given
    "age": members.bd.where(members.bd.between(10, 90)).values,                               # invalid ages become NaN
    "tenure_days": (CUTOFF - reg_date).dt.days.values,
})
del members
member_feats.describe().round(1)

,city,registered_via,gender,age,tenure_days
count,860967.0,860967.0,860967.0,386501.0,860967.0
mean,5.9,6.9,0.7,29.9,1294.7
std,6.4,1.9,0.8,8.8,1097.5
min,1.0,3.0,0.0,10.0,-24.0
25%,1.0,7.0,0.0,24.0,438.0
50%,1.0,7.0,0.0,28.0,1033.0
75%,13.0,9.0,1.0,34.0,1872.0
max,22.0,13.0,2.0,90.0,4753.0


## Transaction Features

In [3]:
tx = pd.read_csv(RAW / "transactions.csv")
tx = tx[tx.msno.isin(uid.index)]
tx["transaction_date"] = pd.to_datetime(tx.transaction_date.astype(str), format="%Y%m%d")
tx["membership_expire_date"] = pd.to_datetime(tx.membership_expire_date.astype(str), format="%Y%m%d")
tx = tx.sort_values(["msno", "transaction_date", "membership_expire_date"])
tx["is_discount"] = (tx.actual_amount_paid < tx.plan_list_price).astype(int)

g = tx.groupby("msno")
tx_feats = g.agg(tx_count=("is_cancel", "size"), total_paid=("actual_amount_paid", "sum"),
                 avg_plan_days=("payment_plan_days", "mean"), auto_renew_ratio=("is_auto_renew", "mean"),
                 cancel_count=("is_cancel", "sum"), discount_count=("is_discount", "sum"),
                 n_payment_methods=("payment_method_id", "nunique"))
last = g.tail(1).set_index("msno")
tx_feats = tx_feats.assign(
    last_payment_method=last.payment_method_id,
    last_plan_days=last.payment_plan_days,
    last_amount_paid=last.actual_amount_paid,
    last_auto_renew=last.is_auto_renew,
    last_is_cancel=last.is_cancel,
    days_since_last_tx=(CUTOFF - last.transaction_date).dt.days,
    days_until_expiration=(last.membership_expire_date - CUTOFF).dt.days,  # negative means already expired
).reset_index()
del tx, g, last
tx_feats.describe().round(1)

,tx_count,total_paid,avg_plan_days,auto_renew_ratio,cancel_count,discount_count,n_payment_methods,last_payment_method,last_plan_days,last_amount_paid,last_auto_renew,last_is_cancel,days_since_last_tx,days_until_expiration
count,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0,933578.0
mean,1.2,175.9,33.7,0.9,0.0,0.0,1.0,38.9,33.9,144.8,0.9,0.0,14.4,23.6
std,1.3,231.8,31.5,0.3,0.2,0.2,0.1,3.8,33.3,140.6,0.3,0.1,23.7,40.5
min,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,-31.0
25%,1.0,99.0,30.0,1.0,0.0,0.0,1.0,38.0,30.0,99.0,1.0,0.0,5.0,9.0
50%,1.0,149.0,30.0,1.0,0.0,0.0,1.0,41.0,30.0,149.0,1.0,0.0,13.0,18.0
75%,1.0,149.0,30.0,1.0,0.0,0.0,1.0,41.0,30.0,149.0,1.0,0.0,22.0,27.0
max,208.0,13650.0,450.0,1.0,8.0,9.0,8.0,41.0,450.0,2000.0,1.0,1.0,819.0,2113.0


## Listening Features

In [4]:
cols = ["msno", "date", "num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]
parts = []
for chunk in pd.read_csv(RAW / "user_logs.csv", usecols=cols, chunksize=3_000_000):
    chunk = chunk[chunk.msno.isin(uid.index)]
    parts.append(pd.DataFrame({
        "uid": uid[chunk.msno].values,
        "day": (chunk.date.values - 20170300).astype(np.int8),
        "secs": chunk.total_secs.clip(0, 86_400).astype(np.float32).values,   # impossible values clipped to one day
        "songs": (chunk.num_25 + chunk.num_50 + chunk.num_75 + chunk.num_985 + chunk.num_100).astype(np.int32).values,
        "unq": chunk.num_unq.astype(np.int32).values,
        "n25": chunk.num_25.astype(np.int32).values,
        "n100": chunk.num_100.astype(np.int32).values,
    }))
logs = pd.concat(parts, ignore_index=True)
del parts
logs["secs_7d"] = np.where(logs.day >= 25, logs.secs, 0)   # last 7 days are 25 to 31 March
logs["active_7d"] = (logs.day >= 25).astype(np.int8)

g = logs.groupby("uid")
L = g.agg(active_days=("day", "nunique"), secs=("secs", "sum"), secs_std=("secs", "std"), songs=("songs", "sum"),
          unq=("unq", "sum"), n25=("n25", "sum"), n100=("n100", "sum"), secs_7d=("secs_7d", "sum"),
          active_days_7d=("active_7d", "sum"), last_day=("day", "max"))
songs = L.songs.replace(0, np.nan)
log_feats = pd.DataFrame({
    "msno": users[L.index.values],
    "active_days": L.active_days,
    "total_hours": L.secs / 3600,
    "secs_per_day": L.secs / L.active_days,
    "songs_per_day": L.songs / L.active_days,
    "unique_song_ratio": (L.unq / songs).fillna(0),
    "skip_ratio": (L.n25 / songs).fillna(0),
    "complete_ratio": (L.n100 / songs).fillna(0),
    "last7_share": (L.secs_7d / L.secs.replace(0, np.nan)).fillna(0),
    "active_days_7d": L.active_days_7d.clip(upper=7),
    "days_since_last_active": 31 - L.last_day,
    "daily_secs_cv": (L.secs_std / (L.secs / L.active_days).replace(0, np.nan)).fillna(0),
}).reset_index(drop=True)
del logs, g, L
log_feats.describe().round(2)

,active_days,total_hours,secs_per_day,songs_per_day,unique_song_ratio,skip_ratio,complete_ratio,last7_share,active_days_7d,days_since_last_active,daily_secs_cv
count,754551.00,754551.00,754551.00,754551.00,754551.00,754551.00,754551.00,754551.00,754551.00,754551.00,754551.00
mean,17.94,38.78,6458.99,33.05,0.77,0.19,0.70,0.22,3.99,2.71,0.77
std,9.68,51.09,6280.66,29.40,0.18,0.15,0.21,0.19,2.48,5.50,0.32
min,1.00,0.00,0.04,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,9.00,8.39,2858.34,16.00,0.69,0.07,0.59,0.10,2.00,0.00,0.60
50%,19.00,22.96,4709.02,25.33,0.82,0.15,0.74,0.20,4.00,0.00,0.76
75%,27.00,48.79,7711.00,40.23,0.90,0.26,0.86,0.29,6.00,2.00,0.95
max,31.00,744.00,86400.00,1987.60,1.00,1.00,1.00,1.00,7.00,30.00,3.67


## Join All Tables

In [5]:
def build(base):
    df = (base.merge(member_feats, on="msno", how="left")
              .merge(tx_feats, on="msno", how="left")
              .merge(log_feats, on="msno", how="left"))
    df["has_member"] = df.city.notna().astype(int)
    df["has_tx"] = df.tx_count.notna().astype(int)
    df["has_logs"] = df.active_days.notna().astype(int)
    zero = ["tx_count", "total_paid", "cancel_count", "discount_count", "n_payment_methods",
            "active_days", "total_hours", "secs_per_day", "songs_per_day", "unique_song_ratio", "skip_ratio",
            "complete_ratio", "last7_share", "active_days_7d", "daily_secs_cv"]
    df[zero] = df[zero].fillna(0)
    df["days_since_last_active"] = df.days_since_last_active.fillna(31)
    num = df.columns.drop(["msno"] + (["is_churn"] if "is_churn" in df else []))
    df[num] = df[num].astype(np.float32)
    return df

train_df, test_df = build(train), build(test)
missing = train_df.isna().mean().mul(100).round(2)
print(train_df.shape, test_df.shape)
print(missing[missing > 0].to_string())

train_df.to_parquet(PROCESSED / "features_train.parquet", index=False, compression="zstd")
test_df.to_parquet(PROCESSED / "features_kaggle_test.parquet", index=False, compression="zstd")

summary = json.loads((RESULTS / "data_summary.json").read_text())
summary["features"] = {"rows": len(train_df), "n_features": train_df.shape[1] - 2,
                       "missing_pct": {k: float(v) for k, v in missing[missing > 0].items()}}
(RESULTS / "data_summary.json").write_text(json.dumps(summary, indent=1, default=float))
train_df.head()

(946686, 35) (24274, 34)
city                     11.32
registered_via           11.32
gender                   11.32
age                      60.19
tenure_days              11.32
avg_plan_days             3.85
auto_renew_ratio          3.85
last_payment_method       3.85
last_plan_days            3.85
last_amount_paid          3.85
last_auto_renew           3.85
last_is_cancel            3.85
days_since_last_tx        3.85
days_until_expiration     3.85


,msno,is_churn,city,registered_via,gender,age,tenure_days,tx_count,total_paid,avg_plan_days,...,unique_song_ratio,skip_ratio,complete_ratio,last7_share,active_days_7d,days_since_last_active,daily_secs_cv,has_member,has_tx,has_logs
0,Pp9fOyd/eG4D7qs1jUnvFagO/9XcbAVNkufTX2Q7wkw=,0,5.0,9.0,2.0,29.0,2028.0,2.0,298.0,30.0,...,0.000000,0.000000,0.000000,0.000000,0.0,31.0,0.000000,1.0,1.0,0.0
1,nUiYMBsiB/HaDWMF20ebBeWtw3xwoWrIGuFWa/J/Adc=,0,13.0,9.0,2.0,36.0,2026.0,1.0,149.0,30.0,...,1.000000,0.083799,0.849162,0.112222,2.0,2.0,0.466059,1.0,1.0,1.0
2,U6POp25OGlgekxcGt8yayx+myCGmO/n71UPqYxxyxQs=,0,13.0,9.0,2.0,40.0,2026.0,1.0,180.0,30.0,...,0.980769,0.050699,0.914336,0.232905,5.0,0.0,0.997823,1.0,1.0,1.0
3,dN8D0n8vugg05OR4U8NHIkd5g7HIc9TTuR003ZdYWR0=,0,3.0,9.0,2.0,46.0,2023.0,1.0,149.0,30.0,...,0.914634,0.014634,0.934146,0.155933,5.0,1.0,0.455429,1.0,1.0,1.0
4,JZn5kjsN5nNtqfUtlpu/+XNYzSsqqiGzejuRLbbBtoE=,0,4.0,9.0,1.0,28.0,2021.0,1.0,149.0,30.0,...,0.675000,0.313158,0.377632,0.138976,4.0,0.0,0.660029,1.0,1.0,1.0
